# DTMFormer 复现 · Kaggle 一键跑通版

**目标**：不训练，直接加载作者发布的 ACDC 权重，跑通推理，拿到 **Dice + HD95 + 参数量 + GFLOPs + 预测可视化图**。

**开始前 3 个设置**（不做完会报错）：

| 设置 | 在哪 | 选什么 |
|---|---|---|
| GPU | 右侧 Settings → Accelerator | **GPU T4** |
| 联网 | 右侧 Settings → Internet | **On**（要 git clone 和下载 137MB 权重） |
| 数据集 | 右侧 Add Input → 搜 `acdc-dataset` | **anhoangvo/acdc-dataset**（324MB，MIT，免申请） |

**使用方式**：从上到下逐个运行 Cell。每一步都有注释，报错就把红色信息整段发给阿澈。

## Cell 1 · 确认环境和数据长什么样

In [ ]:
import os, glob, torch

print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("显卡:", torch.cuda.get_device_name(0))

print("\n/kaggle/input 下的内容：")
for p in sorted(glob.glob("/kaggle/input/*")):
    print(" ", p)
    for q in sorted(glob.glob(p + "/*"))[:5]:
        print("    └", q)

## Cell 2 · 装依赖 + 克隆代码

除了常见库，还要装 2 个容易漏的：`thop`（算参数量/GFLOPs）、`hausdorff`（test.py 顶部 import，不装直接报错）。
`medpy` 用来算 HD95，**装不上也没关系**，Cell 7 里有等价实现兜底。

> ⚠️ **千万不要 `pip install visualizer`**。PyPI 上那个同名包里面没有作者要的 `get_local`，
> 装了反而报错（ImportError: cannot import name 'get_local'）。
> 作者代码里这行 import 是残留——我搜过全仓库，`get_local` 从未被调用过。
> Cell 7 会用一个占位模块顶掉它，对计算结果零影响。

In [ ]:
!pip -q install einops thop hausdorff
!pip -q install medpy || echo "[提示] medpy 没装上，Cell 7 会用等效实现替代，不影响结果"

import os
os.chdir("/kaggle/working")
if not os.path.exists("DTMFormer"):
    !git clone --depth 1 https://github.com/iam-nacl/DTMFormer.git
os.chdir("/kaggle/working/DTMFormer")
print("当前目录:", os.getcwd())
!ls

## Cell 3 · 下载**真正的**权重（这一步最容易翻车）

仓库里的 `checkpoints_ACDC/Setr_DTMFormer.pth` 是 **Git LFS 指针文件，只有 134 字节**，
直接 clone 下来拿去 `torch.load` 必报错。真实文件 **143,838,716 字节（约 137MB）**，
要走 GitHub 的 media 通道下载。下面带大小校验，下错了会直接告诉你。

In [ ]:
import os

CKPT = "checkpoints_ACDC/Setr_DTMFormer.pth"
URL  = "https://media.githubusercontent.com/media/iam-nacl/DTMFormer/main/checkpoints_ACDC/Setr_DTMFormer.pth"
REAL_SIZE = 143838716   # 字节，我已实际下载核对过

if not os.path.exists(CKPT) or os.path.getsize(CKPT) != REAL_SIZE:
    print("开始下载真实权重（约 137MB，1-3 分钟）...")
    !curl -L --retry 3 -o {CKPT} {URL}

size = os.path.getsize(CKPT)
print(f"权重文件大小: {size/1e6:.1f} MB")
assert size == REAL_SIZE, "大小不对，说明还是 LFS 指针或下载被截断，重跑本 Cell"
print("✅ 权重 OK")

## Cell 4 · 把 ACDC 转成 DTMFormer 要的格式

DTMFormer 的数据读取规则（我从 `utils/utils_multi.py` 里核对过）：

```
ACDC/
  img/xxx.png          灰度图
  label/xxx.png        标签，取值 0/1/2/3
  MainPatient/
    trainofficial.txt  训练集清单（只评估也必须存在，test.py 会读）
    testofficial.txt   测试集清单，每行一个不带 .png 的文件名
```

**两条硬约束**：
1. 文件名**前 3 个字符必须是病人编号**，因为 test.py 用 `int(filename[:3])` 把切片归并成病人。
2. txt 里必须按 **病人 → 帧 → 切片序号** 排序，patient 模式靠顺序把 2D 切片拼回 3D 心脏。

> ⚠️ 论文的病人划分没公开，这里用「最后 20 个病人做测试」。
> **所以你的 Dice 不会恰好等于论文的 92.53%**，这是正常的，不是复现失败。

In [ ]:
import os, re, glob, h5py, numpy as np, cv2

SRC = "/kaggle/input"           # Kaggle 挂载的数据集
OUT = "/kaggle/working/ACDC"    # 输出成 DTMFormer 格式

h5s = sorted(glob.glob(os.path.join(SRC, "**", "*_slice_*.h5"), recursive=True))
print("找到切片 h5:", len(h5s))
assert len(h5s) > 0, "没找到 *_slice_*.h5，把 Cell 1 的目录打印发给阿澈"
print("示例路径:", h5s[0])

for d in ["img", "label", "MainPatient"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

IMG_SIZE = 256      # 模型硬要求：见下面「为什么必须是 256」

def norm_u8(x):
    # 把 MRI 灰度归一化到 0-255，去掉极端亮暗点
    x = x.astype(np.float32)
    lo, hi = np.percentile(x, 1), np.percentile(x, 99)
    x = np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)
    return (x * 255).astype(np.uint8)

# 为什么必须是 256×256：
# 1) 模型 Setr_DTMFormer 默认 img_size=256、patch_size=8 →
#    patch_num = 256/8 = 32，后面把 token 还原成二维时就按 32×32 拼。
#    输入不是 256，拼回去的网格尺寸就对不上 / 直接崩。
# 2) README 里那个 --imgsize 256 是障眼法：JointTransform2D 里根本没有无条件 resize，
#    img_size 只用在随机 scale / crop 这些增强里，而 val 模式的 p_scale=0 → 一次都不缩放。
#    ImageToImage2D 读图也只是 cv2.imread(...)，同样不缩放。
# 结论：**磁盘上的 png 必须已经是 256×256**，pipeline 不会帮你缩。
records = []   # (病人号, 帧, 切片号, 新文件名)
skipped = 0
for p in h5s:
    stem = os.path.basename(p)[:-3]                     # patient001_frame01_slice_3
    m = re.match(r"patient(\d+)_(frame\d+)_slice_(\d+)", stem)
    assert m, f"文件名不符合预期: {stem}"
    pid, frame, sid = int(m.group(1)), m.group(2), int(m.group(3))

    with h5py.File(p, "r") as f:
        img, lab = f["image"][:], f["label"][:]
    if lab.max() == 0:          # 没有前景的空白切片跳过，评估更快
        continue

    new_id = f"{pid:03d}_{frame}_slice_{sid:03d}"       # 前3位=病人号
    img_u8 = norm_u8(img)
    img_rs = cv2.resize(img_u8, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    # 标签必须用最近邻：它存的是类别编号 0/1/2/3，双线性插值会造出 1.7 这种不存在的类别
    lab_rs = cv2.resize(lab.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    if lab_rs.max() == 0:      # resize 后万一前景被抹掉，丢掉
        skipped += 1
        continue
    cv2.imwrite(f"{OUT}/img/{new_id}.png", img_rs)
    cv2.imwrite(f"{OUT}/label/{new_id}.png", lab_rs)
    records.append((pid, frame, sid, new_id))

print("有效切片:", len(records), "| 病人数:", len({r[0] for r in records}), "| 因无前景跳过:", skipped)
print("原图 -> 统一后:", img.shape, "->", (IMG_SIZE, IMG_SIZE))
print("标签取值:", np.unique(cv2.imread(f"{OUT}/label/{records[0][3]}.png", 0)))

# 复核一遍，别让尺寸问题拖到第 7 格才发现
_sizes = {cv2.imread(f"{OUT}/img/{r[3]}.png", 0).shape for r in records[:200]}
print("抽查前 200 张尺寸:", _sizes)

## Cell 5 · 划分训练/测试病人，生成两个 txt

In [ ]:
pids = sorted({r[0] for r in records})
N_TEST = 20
test_pids  = set(pids[-N_TEST:])       # 最后 20 个病人做测试
train_pids = set(pids) - test_pids

records.sort(key=lambda r: (r[0], r[1], r[2]))   # 病人→帧→切片，顺序不能乱
train_ids = [r[3] for r in records if r[0] in train_pids]
test_ids  = [r[3] for r in records if r[0] in test_pids]

with open(f"{OUT}/MainPatient/trainofficial.txt", "w") as f:
    f.write("\n".join(train_ids))
with open(f"{OUT}/MainPatient/testofficial.txt", "w") as f:
    f.write("\n".join(test_ids))

print(f"trainofficial: {len(train_ids)} 张 / {len(train_pids)} 人（只是占位，不训练）")
print(f"testofficial : {len(test_ids)} 张 / {len(test_pids)} 人  ← 实际评估这些")
print("测试集前 3 个文件名:", test_ids[:3])

## Cell 6 · 瞄一眼数据对不对（10 秒，能省半小时）

In [ ]:
import matplotlib.pyplot as plt

sid = test_ids[len(test_ids)//2]
img = cv2.imread(f"{OUT}/img/{sid}.png", 0)
lab = cv2.imread(f"{OUT}/label/{sid}.png", 0)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img, cmap="gray"); ax[0].set_title(f"image: {sid}"); ax[0].axis("off")
ax[1].imshow(lab, vmin=0, vmax=3); ax[1].set_title("label: 0背景 1RV 2MYO 3LV"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print("如果左边是心脏 MRI、右边能看到 3 个彩色结构，说明数据转换成功")

## Cell 7 · 正式跑评估：Dice + HD95 + 参数量 + GFLOPs

两个补丁，都是为了绕过原作者代码在新环境里的问题：
- **补丁0**：作者代码 import 了 `visualizer.get_local`，但整个仓库从未调用，PyPI 同名包又没有这个函数 → 塞占位模块。
- **补丁1**：PyTorch ≥ 2.6 的 `torch.load` 默认 `weights_only=True`，老 checkpoint 会加载失败。
- **补丁2**：`medpy` 装不上时，用 scipy 写一个**公式相同**的 hd95。

补丁 1 特意写成**可以重复运行**的形式。踩过一次的坑：如果写成
`_raw_load = torch.load` 之后又重跑这格，`_raw_load` 会变成上一轮的补丁函数，
于是补丁调补丁无限套娃 → `RecursionError`。

用 `runpy` 而不是 `!python test.py`，这样补丁才能生效。

In [ ]:
import os, sys, runpy, types, numpy as np, torch

CKPT_ABS = os.path.abspath("checkpoints_ACDC/Setr_DTMFormer.pth")

# ---- 补丁 0：visualizer 占位 ----
# 作者的 model_utils.py 里 from visualizer import get_local，
# 但整个仓库从未真正调用它。PyPI 上装到的 visualizer 是同名不同包，用不了，
# 所以直接塞一个占位模块，对计算结果零影响。
_vis = types.ModuleType("visualizer")
def get_local(*args, **kwargs):
    if len(args) == 1 and callable(args[0]):
        return args[0]                       # 裸装饰器用法
    def _decorator(fn):
        return fn                            # 带参装饰器用法
    return _decorator
_vis.get_local = get_local
sys.modules["visualizer"] = _vis
print("visualizer 占位模块已就位")

# ---- 补丁 1：老 checkpoint 兼容（可重复运行，不会自我套娃）----
# 坑：如果写成 _raw_load = torch.load 再重跑这格，_raw_load 会变成上一轮的补丁函数，
#     于是补丁调补丁、补丁调补丁 → RecursionError。下面两种写法都能避免：
def _unwrap(fn, max_depth=32):
    # torch.load 若已被打过补丁，顺着闭包链找到最里层的原始函数
    for _ in range(max_depth):
        clo = getattr(fn, "__closure__", None)
        if not clo:
            break
        nxt = None
        for c in clo:
            v = c.cell_contents
            if callable(v) and v is not fn:
                nxt = v
                break
        if nxt is None or nxt is fn:
            break
        fn = nxt
    return fn

# 优先直接拿 torch.serialization.load —— 它是 PyTorch 真正的原始实现，
# 引用它而不是 torch.load，就彻底断掉了自引用的可能。
_TRUE_LOAD = getattr(torch.serialization, "load", None) or _unwrap(torch.load)

def _patched_load(*a, _real=_TRUE_LOAD, **kw):
    # _real 用默认参数在「定义时」就把真函数绑死。
    # 为什么不用全局名？在 notebook 里 _patched_load 每次执行都会重新触发全局赋值，
    # 而函数体里的全局名是**运行时**才去查的 —— 上一轮留下的补丁也会跟着指向新的全局，
    # 于是 P1 调 P1，直接 RecursionError。默认参数把这个坑彻底焊死。
    kw.setdefault("weights_only", False)
    return _real(*a, **kw)
torch.load = _patched_load

# 自检：确认补丁没有自引用。出问题就直接报人话，而不是甩你一个 RecursionError。
assert _TRUE_LOAD is not torch.load, "补丁指向了自己，请 Kernel → Restart 后从第 1 格重跑"
_f, _depth = torch.load, 0
while getattr(_f, "__closure__", None):
    cands = [c.cell_contents for c in _f.__closure__
             if callable(c.cell_contents) and c.cell_contents is not _f]
    if not cands:
        break
    _f, _depth = cands[0], _depth + 1
    if _depth > 3:
        raise RuntimeError("torch.load 被重复打补丁套娃了，请 Kernel → Restart 后从第 1 格重跑")
print("torch.load 补丁已就位并通过自检（可随意重跑，不会递归）")

# ---- 补丁 2：medpy 缺失时的等效 hd95 ----
try:
    import medpy
    print("medpy 可用，用原版实现")
except Exception as e:
    print("medpy 不可用，启用等效实现")
    from scipy.ndimage import distance_transform_edt, binary_erosion

    def hd95(pred, gt, voxelspacing=None, connectivity=1):
        pred, gt = pred.astype(bool), gt.astype(bool)
        if pred.sum() == 0 or gt.sum() == 0:
            return float("nan")
        def one_way(a, b):
            dt = distance_transform_edt(~(a ^ binary_erosion(a)), sampling=voxelspacing)
            return dt[b ^ binary_erosion(b)]
        d = np.concatenate([one_way(pred, gt), one_way(gt, pred)])
        return float(np.percentile(d, 95))

    m_medpy  = types.ModuleType("medpy")
    m_metric = types.ModuleType("medpy.metric")
    m_binary = types.ModuleType("medpy.metric.binary")
    m_binary.hd95 = hd95
    m_metric.binary = m_binary
    m_medpy.metric = m_metric
    sys.modules.update({"medpy": m_medpy, "medpy.metric": m_metric,
                        "medpy.metric.binary": m_binary})

# ---- 跑原作者的 test.py，一个字符都不改 ----
sys.argv = ["test.py",
            "--dataset", "/kaggle/working/ACDC",
            "--loaddirec", CKPT_ABS,
            "--eval_mode", "patient",   # patient 模式才算真正的 HD95
            "--classes", "4"]
runpy.run_path("test.py", run_name="__main__")

### 怎么看输出

```
GFLOPS: 9.xx           ← 计算量，论文报 9.8
参数量: xx M           ← 参数量
mdices  [0.  d1 d2 d3] ← RV / MYO / LV 三类各自的 Dice（0 是背景占位，忽略）
mhds    [0.  h1 h2 h3] ← 三类各自的 HD95，单位是像素，越小越好
dices / hds ...        ← 最后一组是三类平均，**这个平均 Dice 就是要报的数字**
```

论文 ACDC 平均 Dice = **92.53%**。你这边病人划分不同、预处理不同，
**落在 88~93% 都完全正常**；如果低于 80%，那才有问题，把输出发我。

## Cell 8 · 出预测可视化图（邮件/笔记能用，可跳过）

In [ ]:
import matplotlib.pyplot as plt
from models import SETR
from utils.utils_multi import JointTransform2D

device = "cuda"
model = SETR.Setr_DTMFormer(classes=4).to(device)
model.load_state_dict(torch.load(CKPT_ABS, map_location=device, weights_only=False))
model.eval()

tf = JointTransform2D(img_size=256, crop=None, p_flip=0, p_gama=0,
                      color_jitter_params=None, long_mask=True)

show = test_ids[::max(len(test_ids)//4, 1)][:4]
fig, axes = plt.subplots(len(show), 3, figsize=(10, 3.2*len(show)))

with torch.no_grad():
    for r, name in enumerate(show):
        im = cv2.imread(f"/kaggle/working/ACDC/img/{name}.png", 0)
        gt = cv2.imread(f"/kaggle/working/ACDC/label/{name}.png", 0)
        x, y = tf(im, gt)
        out, _ = model(x.unsqueeze(0).float().to(device))
        pred = out.softmax(1).argmax(1)[0].cpu().numpy()

        axes[r,0].imshow(x[0].cpu(), cmap="gray"); axes[r,0].set_title(name, fontsize=8)
        axes[r,1].imshow(y.cpu(), vmin=0, vmax=3); axes[r,1].set_title("真实标签 GT")
        axes[r,2].imshow(pred, vmin=0, vmax=3);    axes[r,2].set_title("DTMFormer 预测")
        for c in range(3): axes[r,c].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/dtmformer_predictions.png", dpi=130)
plt.show()
print("图片已存到 /kaggle/working/dtmformer_predictions.png")

## Cell 9 · 把结果记下来（复制到你的笔记本/邮件）

填这张表，就是今天复现的完整交付物：

| 项目 | 你的结果 | 论文参考 |
|---|---|---|
| 平均 Dice（3 类） | ___ % | 92.53% |
| 平均 HD95 | ___ 像素 | — |
| 参数量 | ___ M | — |
| GFLOPs | ___ | 9.8 |
| 测试病人数 / 切片数 | 20 人 / ___ 张 | 未公开 |

**邮件里可以这样写（诚实口径，别写"复现了论文结果"）**：

> 我在 Kaggle T4 上跑通了您 DTMFormer 的官方代码：加载作者发布的 ACDC 权重，
> 在自建的 20 例病人测试集上完成 patient 级评估，输出平均 Dice 与 HD95，
> 并核对了模型的参数量与 GFLOPs。由于论文未公开病人划分，数值与论文略有差异。

## Cell 10 · 兜底：数据实在搞不定时的最小验证

只验证「模型能建起来、权重能加载、前向能跑、参数量/GFLOPs 能算」。
**不需要任何数据集**，10 分钟能完成。有这个也足够在邮件里写"已跑通模型前向与权重加载"。

In [ ]:
import torch
from thop import profile
from models import SETR

model = SETR.Setr_DTMFormer(classes=4).cuda()
model.load_state_dict(torch.load(CKPT_ABS, map_location="cuda", weights_only=False))
model.eval()
print("✅ 权重加载成功")

x = torch.randn(1, 1, 256, 256).cuda()
flops, params = profile(model, inputs=(x,))
print(f"GFLOPs: {flops/1e9:.2f} | 参数量: {params/1e6:.2f} M")

with torch.no_grad():
    out, _ = model(x)
print("输出形状:", tuple(out.shape), "← 应为 (1, 4, 256, 256)")
print("说明：256/8 = 32，即 32×32 = 1024 个 token，和你 DINOv2 448 输入时的 token 数一样")

## 第 10 格 · 预处理诊断（可选，用于追 Dice）

第 7 格能跑通并输出 Dice/HD95，但数字低于论文时，用这一格反向搜索最合适的输入构造：
4 种亮度方案 x 3 种几何方案网格搜索 -> 缩放搜索 -> 混淆矩阵与可视化。
作者代码全程没有归一化（z_score 默认 False，train/test 一致），所以训练期的像素约定被烙在权重里。

### 第 11 格 · 空间错位检查（可选）

第 10 格若显示所有预处理组合的 Dice 都挤在同一水平，说明亮度和几何不是主因。
这一格改查**位置**：对真值做整数平移和翻转，看能不能把 Dice 顶上去；同时出可视化图。

In [ ]:
# ============================================================
# 收尾检查：修掉 zoom 越界 + 查空间错位 + 出图
# 前提：上一个诊断格已经在同一个 kernel 里跑过（frames / model 都在）
# ============================================================
import numpy as np, cv2, torch, matplotlib.pyplot as plt

assert "frames" in globals() and "model" in globals(), "先跑上一个诊断格"

def to256(im, geo, zoom=1.0, nearest=False):
    """把一张图摆进 256x256 画布。修掉了内容超出画布时 copyMakeBorder 负数报错的问题。"""
    S = 256
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR
    h, w = im.shape
    if geo == "stretch":
        return cv2.resize(im, (S, S), interpolation=interp)
    if geo == "pad":
        s = zoom * S / max(h, w)
        nh, nw = max(int(round(h * s)), 1), max(int(round(w * s)), 1)
        r = cv2.resize(im, (nw, nh), interpolation=interp)
        if nh > S or nw > S:                       # 内容比画布大 -> 中心裁回来
            y0 = max((nh - S) // 2, 0)
            x0 = max((nw - S) // 2, 0)
            return r[y0:y0 + S, x0:x0 + S]
        t, b = (S - nh) // 2, S - nh - (S - nh) // 2
        l, rt = (S - nw) // 2, S - nw - (S - nw) // 2
        return cv2.copyMakeBorder(r, t, b, l, rt, cv2.BORDER_CONSTANT, value=0)
    if geo == "crop":
        side = min(max(int(min(h, w) / zoom), 1), h, w)
        y0, x0 = (h - side) // 2, (w - side) // 2
        return cv2.resize(im[y0:y0 + side, x0:x0 + side], (S, S), interpolation=interp)
    raise ValueError(geo)

@torch.no_grad()
def predict(x8):
    t = torch.from_numpy(x8[None, None].astype(np.float32) / 255.0).to(device)
    out, _ = model(t)
    return out.softmax(1).argmax(1)[0].cpu().numpy().astype(np.uint8)

def dice_list(p, g):
    ds = []
    for c in (1, 2, 3):
        a, b = (p == c), (g == c)
        if a.sum() == 0 and b.sum() == 0:
            continue
        if a.sum() == 0 or b.sum() == 0:
            ds.append(0.0); continue
        ds.append(2 * (a & b).sum() / (a.sum() + b.sum()))
    return float(np.mean(ds)) if ds else float("nan")

def evaluate(how, geo, zoom=1.0):
    vals = []
    for key, vol, lab in frames:
        im8 = intensity(vol, how)
        for i in range(len(im8)):
            if lab[i].max() == 0:
                continue
            x = to256(im8[i], geo, zoom=zoom)
            y = to256(lab[i], geo, zoom=zoom, nearest=True)
            vals.append(dice_list(predict(x), y))
    return float(np.nanmean(vals))

# ---------- 1) 补完整第二轮：这次 zoom>1 不会炸了 ----------
print("=" * 62)
print("第二轮（补完）：内容缩放搜索，固定 亮度=perc")
print("=" * 62)
best = (0.6117, "stretch", 1.0)
for geo in ["pad", "crop"]:
    for z in [0.85, 0.95, 1.05, 1.15, 1.30, 1.50]:
        d = evaluate("perc", geo, zoom=z)
        flag = ""
        if d > best[0]:
            best = (d, geo, z)
            flag = "   <== 刷新最优"
        print("  %-6s zoom=%.2f  Dice=%.4f%s" % (geo, z, d, flag))
print("\n缩放搜索最优: geo=%s zoom=%.2f  Dice=%.4f" % (best[1], best[2], best[0]))

# ---------- 2) 空间错位检查：既然预处理不是主因，那就看是不是"位置对不上" ----------
print("\n" + "=" * 62)
print("空间错位检查：预测和标签是否在位置上错开")
print("=" * 62)

how, geo, zoom = "perc", best[1], best[2]
cases = []
for key, vol, lab in frames[:4]:
    im8 = intensity(vol, how)
    for i in range(0, len(im8), 3):
        if lab[i].max() == 0:
            continue
        x = to256(im8[i], geo, zoom=zoom)
        y = to256(lab[i], geo, zoom=zoom, nearest=True)
        cases.append((predict(x), y))
print("用于检查的切片数:", len(cases))

base = float(np.nanmean([dice_list(p, g) for p, g in cases]))
print("不移动时的 Dice: %.4f" % base)

best_shift, best_d = (0, 0), base
for dy in range(-8, 9, 2):
    for dx in range(-8, 9, 2):
        if dy == 0 and dx == 0:
            continue
        d = float(np.nanmean([dice_list(p, np.roll(np.roll(g, dy, 0), dx, 1))
                              for p, g in cases]))
        if d > best_d:
            best_d, best_shift = d, (dy, dx)
print("最佳平移 dy,dx = %s 时 Dice = %.4f" % (str(best_shift), best_d))
if best_d > base + 0.05:
    print(">>> 明显更好！预测和真值在空间上错开了 %s 像素" % str(best_shift))
else:
    print(">>> 没有明显改善，预测位置是对的，不存在空间错位")

for name, fn in [("左右翻转", lambda a: a[:, ::-1]),
                 ("上下翻转", lambda a: a[::-1, :]),
                 ("旋转180度", lambda a: a[::-1, ::-1])]:
    d = float(np.nanmean([dice_list(p, fn(g)) for p, g in cases]))
    print("把真值%s后Dice = %.4f   %s" % (name, d, "↑↑ 这才是病因" if d > base + 0.05 else ""))

# ---------- 3) 出图：这次一定存下来，肉眼定性最重要 ----------
NSHOW = min(6, len(cases))
fig, axes = plt.subplots(NSHOW, 3, figsize=(12, 3.6 * NSHOW))
if NSHOW == 1:
    axes = axes[None, :]
for r in range(NSHOW):
    p, g = cases[r * max(len(cases) // NSHOW, 1)]
    axes[r, 0].imshow(p > 0, cmap="gray")
    axes[r, 0].set_title("预测前景（只看有没有分割出来）")
    axes[r, 1].imshow(g, vmin=0, vmax=3)
    axes[r, 1].set_title("真实标签 GT")
    axes[r, 2].imshow(p, vmin=0, vmax=3)
    axes[r, 2].set_title("DTMFormer 预测")
    for c in range(3):
        axes[r, c].axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/check_align.png", dpi=130)
plt.show()
print("\n已存到 /kaggle/working/check_align.png —— 打开这张图看一眼，比任何数字都直观")